In [4]:
import SimpleITK as sitk
import nibabel as nib
import nii

In [5]:
sitk_im = sitk.ReadImage("origin_spacing.nii.gz")
sitk_arr = sitk.GetArrayFromImage(sitk_im)
spacing = sitk_im.GetSpacing()
print("SITK: Direction ", sitk_im.GetDirection())
print("SITK: Origin ", sitk_im.GetOrigin())
print("SITK: Spacing ",sitk_im.GetSpacing())
print("SITK: Size ",sitk_im.GetSize())

SITK: Direction  (0.9937812403056384, 0.09786909716488885, -0.05310824685933352, -0.08252307991898211, 0.3271237946010296, -0.9413713168750709, 0.07475819485027804, -0.939899815313288, -0.333165961467289)
SITK: Origin  (-146.30685424804688, 68.21996307373047, 136.22999572753906)
SITK: Spacing  (1.0099999904632568, 1.0199999809265137, 1.0299999713897705)
SITK: Size  (256, 256, 256)


In [6]:
im = nii.read_image("origin_spacing.nii.gz")
print("NII: Direction ", im.get_direction())
print("NII: Origin ", im.get_origin())
print("NII: Spacing ",im.get_spacing())
print("NII: Size ", im.get_size())
print("NII: Affine ", im.get_affine())

NII: Direction  [[0.9937812403056384, 0.09786909716488885, -0.05310824685933352], [-0.08252307991898211, 0.3271237946010296, -0.9413713168750709], [0.07475819485027804, -0.939899815313288, -0.333165961467289]]
NII: Origin  [-146.30685424804688, 68.21996307373047, 136.22999572753906]
NII: Spacing  [1.0099999904632568, 1.0199999809265137, 1.0299999713897705]
NII: Size  [256, 256, 256]
NII: Affine  [[-1.00371897e+00 -9.98264775e-02  5.47014922e-02  1.46306854e+02]
 [ 8.33483040e-02 -3.33666265e-01  9.69612420e-01 -6.82199631e+01]
 [ 7.55057707e-02 -9.58697796e-01 -3.43160927e-01  1.36229996e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [18]:
im = nii.read_image("origin_spacing.nii.gz")
im.set_direction([[0,1,0],[0,0,1],[1,0,0]])
print(im.get_direction())

[[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0]]


In [6]:
nib_im = nib.load("origin_spacing.nii.gz")
print(nib_im.affine)

[[-1.00371897e+00 -9.98264775e-02  5.47014922e-02  1.46306854e+02]
 [ 8.33483040e-02 -3.33666265e-01  9.69612420e-01 -6.82199631e+01]
 [ 7.55057707e-02 -9.58697796e-01 -3.43160927e-01  1.36229996e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [7]:
nib_im.header.get_zooms()

(1.01, 1.02, 1.03)

In [16]:
import numpy as np
np.testing.assert_equal(sitk_arr, im.ndarray())

In [4]:
import nibabel as nib
import numpy as np

# 请将 'your_file.nii.gz' 替换为你的文件名
file_path = 'origin_spacing.nii.gz'

try:
    img = nib.load(file_path)
    
    # 打印 sform 和 qform 仿射矩阵
    # NIfTI 标准中有两种仿射矩阵，通常 sform 更常用且更准确
    print("sform_code:", img.header['sform_code'])
    print("sform affine:\n", img.get_sform())
    
    print("\nqform_code:", img.header['qform_code'])
    print("qform affine:\n", img.get_qform())

    # 打印头文件中的体素间距信息
    zooms = img.header.get_zooms()
    print("\nVoxel spacings (pixdim):", zooms)

    # 我们手动计算一下方向余弦
    # 从仿射矩阵中分离出旋转和缩放部分 (3x3)
    affine = img.get_sform()
    rotation_scaling_matrix = (affine[:3, :3]).astype(np.float32)
    
    # 分离出体素间距（缩放因子）
    # 注意，这里假设没有切变(shear)，在医学图像中通常成立
    spacings = np.sqrt(np.sum(rotation_scaling_matrix**2, axis=0))
    print("Calculated spacings from affine:", spacings)

    # 通过除以 spacing 来归一化，得到纯旋转矩阵
    direction_cosines = rotation_scaling_matrix / spacings
    print("\nCalculated direction cosines (from sform):\n", direction_cosines)

except Exception as e:
    print(f"An error occurred: {e}")

sform_code: 1
sform affine:
 [[-1.00371897e+00 -9.98264775e-02  5.47014922e-02  1.46306854e+02]
 [ 8.33483040e-02 -3.33666265e-01  9.69612420e-01 -6.82199631e+01]
 [ 7.55057707e-02 -9.58697796e-01 -3.43160927e-01  1.36229996e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

qform_code: 1
qform affine:
 [[-1.00371919e+00 -9.98254068e-02  5.47007203e-02  1.46306854e+02]
 [ 8.33472465e-02 -3.33666363e-01  9.69612494e-01 -6.82199631e+01]
 [ 7.55050218e-02 -9.58697871e-01 -3.43160871e-01  1.36229996e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

Voxel spacings (pixdim): (1.01, 1.02, 1.03)
Calculated spacings from affine: [1.0099999 1.02      1.03     ]

Calculated direction cosines (from sform):
 [[-0.99378127 -0.0978691   0.05310825]
 [ 0.08252309 -0.3271238   0.9413713 ]
 [ 0.0747582  -0.9398998  -0.33316594]]
